In [ ]:
!pip install transformers sentencepiece --quiet

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-large"  # instruction-tuned
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Safety filter
def is_safe(query):
    forbidden = ["suicide", "inject", "poison", "self-harm", "kill", "dangerous", "overdose"]
    return not any(word in query.lower() for word in forbidden)

def get_health_response(query):
    if not is_safe(query):
        return "Sorry, I cannot provide advice on dangerous topics. Consult a professional."

    prompt = f"You are a friendly medical assistant. Answer the following safely:\n{query}"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(**inputs, max_new_tokens=200)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

# Interactive loop
print("💬 Health Chatbot (type 'exit' to quit)")
while True:
    q = input("\nYou: ")
    if q.lower() in ["exit", "quit"]:
        print("Chatbot: Take care! Consult a doctor if needed.")
        break
    print("Chatbot:", get_health_response(q))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

💬 Health Chatbot (type 'exit' to quit)

You: what causes sore throat
Chatbot: a cold

You: is phracitamol safe for children
Chatbot: yes

You: how can i improve sleep
Chatbot: Drink a glass of water before bed.

You: exit
Chatbot: Take care! Consult a doctor if needed.
